# DDPM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.datasets import MNIST
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML
from tqdm import tqdm
from dlprog import train_progress

prog = train_progress()

plt.style.use("ggplot")
plt.rcParams["animation.embed_limit"] = 30.0 * 1024 * 1024

def show_images(imgs, ncol=5, w=128):
    _, _, h_img, w_img = imgs.shape
    h = int(h_img * w / w_img)
    imgs = transforms.Resize((h, w), antialias=True)(imgs)
    img = torchvision.utils.make_grid(imgs, nrow=ncol)
    img = transforms.functional.to_pil_image(img)
    display(img)

device = torch.accelerator.current_accelerator(check_available=True)
device

## MNIST Dataset

In [ ]:
SIZE = 28
batch_size = 64

ds = MNIST(root="./data", download=True, transform=transforms.ToTensor())
dataloader = DataLoader(ds, batch_size=batch_size, shuffle=True)
x, _ = next(iter(dataloader))
print(f"Number of images: {len(ds)}")
print(f"Image shape: {x.shape}")
show_images(x[:20])

## Noise Schedule

今回はLinearを採用

In [ ]:
T = 100
beta_1 = 1e-4
beta_T = 0.1
beta = torch.linspace(beta_1, beta_T, T + 1)
alpha = 1 - beta
bar_alpha = torch.cumprod(alpha, dim=0)
t = torch.arange(1, T + 1)
t_1 = t - 1
plt.plot(t, beta[t], label=r"$\beta_t$")
plt.plot(t, alpha[t], label=r"$\alpha_t$")
plt.plot(t, bar_alpha[t], label=r"$\bar \alpha_t$");
plt.xlabel(r"$t$")
plt.legend();

In [ ]:
T = 100
s = 0.008
t_ = torch.arange(T + 1)
t = torch.arange(1, T + 1)
t_1 = t - 1
bar_alpha = (torch.cos((t_ / T + s) / (1 + s) * np.pi / 2) ** 2) / (np.cos(s / (1 + s) * np.pi / 2) ** 2)
alpha = bar_alpha[t] / bar_alpha[t_1]
beta = 1 - alpha
tilde_beta = (1 - bar_alpha[t_1]) / (1 - bar_alpha[t]) * beta
bar_alpha = bar_alpha[t]
plt.plot(t, alpha, label=r"$\alpha_t$")
plt.plot(t, bar_alpha, label=r"$\bar \alpha_t$");
plt.plot(t, tilde_beta, label=r"$\tilde \beta_t$");
plt.xlabel(r"$t$")
plt.legend();

## U-Net

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, d=128, max_len=T, base=10_000):
        super().__init__()
        p = torch.arange(max_len)
        i = torch.arange(0, d, 2)
        pe = torch.zeros(max_len, d)
        pe[:, ::2] = torch.sin(p.unsqueeze(-1) / base ** (i / d))
        pe[:, 1::2] = torch.cos(p.unsqueeze(-1) / base ** (i / d))
        self.register_buffer("pe", pe)

    def forward(self, t):
        return self.pe[t]


class UNet(nn.Module):
    def __init__(self, d_t=128):
        super().__init__()
        self.embed = TimeEmbedding(d=d_t)
        self.conv1 = self.get_conv_block(1, 32) # 28 -> 14
        self.conv2 = self.get_conv_block(32, 64) # 14 -> 7
        self.conv3 = self.get_conv_block(64, 128, 3, 1, 0) # 7 -> 5

        self.convT3 = self.get_convT_block(128, 64, 3, 1, 0, 0) # 5 -> 7
        self.convT2 = self.get_convT_block(64 * 2, 32) # 7 -> 14
        self.convT1 = self.get_convT_block(32 * 2, 16) # 14 -> 28

        self.out_conv = nn.Conv2d(16, 1, 1) # 28 -> 28

        self.fc_t1 = nn.Linear(d_t, 32)
        self.fc_t2 = nn.Linear(d_t, 64)
        self.fc_t3 = nn.Linear(d_t, 128)
        self.fc_t4 = nn.Linear(d_t, 128)
        self.fc_t5 = nn.Linear(d_t, 64)
        self.fc_t6 = nn.Linear(d_t, 16)

    def get_conv_block(self, c_in, c_out, kernel_size=7, stride=2, padding=3):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size, stride, padding),
            nn.BatchNorm2d(c_out),
            nn.SiLU(),
        )

    def get_convT_block(self, c_in, c_out, kernel_size=7, stride=2, padding=3, output_padding=1):
        return nn.Sequential(
            nn.ConvTranspose2d(c_in, c_out, kernel_size, stride, padding, output_padding=output_padding),
            nn.BatchNorm2d(c_out),
            nn.SiLU(),
        )

    def forward(self, x, t):
        # x: (b, 1, 28, 28)
        # t: (b,)
        t_embed = self.embed(t)

        z1 = self.conv1(x) # (b, 32, 14, 14)

        z = z1 + self.fc_t1(t_embed)[:, :, None, None]
        z2 = self.conv2(z) # (b, 64, 7, 7)

        z = z2 + self.fc_t2(t_embed)[:, :, None, None]
        z3 = self.conv3(z) # (b, 128, 5, 5)

        z = z3 + self.fc_t3(t_embed)[:, :, None, None]
        z = self.convT3(z) # (b, 64, 7, 7)

        z = torch.cat([z, z2], dim=1) + self.fc_t4(t_embed)[:, :, None, None]
        z = self.convT2(z) # (b, 32, 14, 14)

        z = torch.cat([z, z1], dim=1) + self.fc_t5(t_embed)[:, :, None, None]
        z = self.convT1(z) # (b, 16, 28, 28)

        z = z + self.fc_t6(t_embed)[:, :, None, None]
        z = self.out_conv(z) # (b, 1, 28, 28)

        return z

テスト

In [ ]:
model = UNet().to(device)
x, _ = next(iter(dataloader))
t = torch.randint(0, T, (len(x),))
x = x.to(device)
with torch.no_grad():
    z = model(x, t)
print(z.shape)

## Inference

In [ ]:
@torch.inference_mode()
def sample(model, n_samples):
    model.eval()
    x = torch.randn(n_samples, 1, SIZE, SIZE).to(device)
    hist = [x.cpu()]

    for t in tqdm(range(T, 0, -1), desc="Sampling"):
        t -= 1
        t = torch.tensor([t] * n_samples)
        beta_t = beta[t][:, None, None, None].to(device)
        alpha_t = alpha[t][:, None, None, None].to(device)
        bar_alpha_t = bar_alpha[t][:, None, None, None].to(device)
        t = t.to(device)

        eps = model(x, t)
        mu = (1 / alpha_t.sqrt()) * (x - (1 - alpha_t)/(1 - bar_alpha_t).sqrt() * eps)
        if t[0] == 0:
            noise = 0
        else:
            noise = torch.randn_like(x)
        x = mu + beta_t.sqrt() * noise # 簡略化のためにtilde betaをbetaで近似
        hist.append(x.cpu())
    return x, hist


def diffusion(model, n_samples=10, duration=5):
    _, hist = sample(model, n_samples)
    fig, ax = plt.subplots(figsize=(10, 1))
    ims = []
    for x in hist:
        x = torch.clamp(x, 0, 1)
        im = ax.imshow(torchvision.utils.make_grid(x, nrow=10).permute(1, 2, 0))
        ax.axis("off")
        ims.append([im])
    ani = animation.ArtistAnimation(fig, ims, interval=int(duration*1000/len(hist)), repeat=False)
    display(HTML(ani.to_jshtml()))
    plt.close()

テスト

In [ ]:
diffusion(model)

## Training

In [ ]:
weights = torch.arange(T, dtype=torch.float) + T // 2
weights = weights / weights.sum()

def train(model, dataloader, optimizer, n_epochs=10):
    model.train()
    prog.start(n_iter=len(dataloader), n_epochs=n_epochs, label="loss")
    for _ in range(n_epochs):
        for x0, _ in dataloader:
            optimizer.zero_grad()
            x0 = x0.to(device)
            eps = torch.randn(x0.size()).to(device)

            # t = torch.randint(0, T, (len(x0),))
            t = torch.multinomial(weights, num_samples=len(x0), replacement=True)

            bar_alpha_t = bar_alpha[t, None, None, None].to(device)
            xt = bar_alpha_t.sqrt() * x0 + (1 - bar_alpha_t).sqrt() * eps
            pred = model(xt, t)
            loss = F.mse_loss(pred, eps)
            loss.backward()
            optimizer.step()
            prog.update(loss.item())

In [ ]:
model = UNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
train(model, dataloader, optimizer, n_epochs=5)

In [ ]:
diffusion(model)